# FitVTON: бесплатный пробный запуск

Ноутбук запускает FitVTON в Google Colab на доступной облачной GPU. Он проверяет одно фото человека с фото пальто и брюк. Запуск может завершиться ошибкой нехватки видеопамяти или недоступности GPU; бесплатная сессия не является постоянным сервером.

FitVTON принимает категории типа фигуры, роста, длины и вида вещи. Он не принимает точные сантиметры тела или размер вещи S/M/L. Этот пилот проверяет визуальный результат и реакцию на категории, а не точность размерной сетки.

Перед запуском выберите в Colab среду с GPU. Если базовая FLUX-модель требует доступа, примите условия доступа к ней на Hugging Face и добавьте токен `HF_TOKEN` в раздел Secrets Colab. Не вставляйте токен в ячейки и не отправляйте его в чат.

In [ ]:
import shutil
import subprocess

try:
    import torch
except ImportError as error:
    raise RuntimeError("Не удалось проверить GPU. В Colab выберите Среда выполнения → Сменить среду выполнения → GPU и подключитесь заново.") from error

if not torch.cuda.is_available():
    raise RuntimeError("GPU не подключена. В Colab выберите Среда выполнения → Сменить среду выполнения → Аппаратный ускоритель → GPU, сохраните и подключитесь заново.")

device = torch.cuda.get_device_properties(0)
print(f"GPU: {device.name}; видеопамять: {device.total_memory / 1024**3:.1f} ГБ")
if shutil.which("nvidia-smi"):
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    print(result.stdout)

## Установка

Установка загружает зависимости FitVTON и фиксирует репозиторий на проверенной версии. Она может занять несколько минут и скачать несколько гигабайт.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path("/content/FitVTON")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ZenoNing/FitVTON.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", "17078c64cc6f6984466bc31916e40421d97f3280"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "17078c64cc6f6984466bc31916e40421d97f3280"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--index-url", "https://download.pytorch.org/whl/cu126", "torch==2.11.0+cu126", "torchvision==0.26.0+cu126", "torchaudio==2.11.0+cu126"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "diffusers==0.38.0", "transformers==5.10.2", "accelerate==1.13.0", "peft>=0.18.0", "bitsandbytes==0.47.0", "huggingface-hub>=1.5.0,<2.0", "safetensors>=0.8.0", "tokenizers>=0.22.0,<=0.23.0", "sentencepiece==0.2.1", "protobuf>=6.33.6", "einops==0.8.1", "numpy==2.2.6", "Pillow==11.3.0"], check=True)
print("Зависимости установлены.")

## Подготовка официального скрипта

В проверенной версии репозитория параметр `--height` объявлен дважды: один раз как высота изображения, второй — как категория роста человека. Ячейка ниже переименовывает только параметр высоты изображения, чтобы команды запуска работали.

In [ ]:
import diffusers
import shutil

pipeline_source = REPO / "pipeline_flux_kontext_multiple_images.py"
pipeline_target = Path(diffusers.__file__).resolve().parent / "pipelines/flux/pipeline_flux_kontext_multiple_images.py"
pipeline_target.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(pipeline_source, pipeline_target)

inference_script = REPO / "inference_demo.py"
script = inference_script.read_text(encoding="utf-8")
old_resolution_arg = 'parser.add_argument("--height", type=int, default=1024)'
if old_resolution_arg not in script:
    raise RuntimeError("Не найден ожидаемый параметр высоты изображения; версия скрипта изменилась.")
script = script.replace(old_resolution_arg, 'parser.add_argument("--image_height", type=int, default=1024)', 1)
if script.count("height=args.height,") != 1:
    raise RuntimeError("Не найден единственный вызов высоты изображения; запуск остановлен для проверки.")
script = script.replace("height=args.height,", "height=args.image_height,", 1)
if script.count('"height": args.height,') != 1:
    raise RuntimeError("Не найдено ожидаемое поле размера результата; запуск остановлен для проверки.")
script = script.replace('"height": args.height,', '"height": args.image_height,', 1)
inference_script.write_text(script, encoding="utf-8")
subprocess.run([sys.executable, str(inference_script), "--help"], cwd=REPO, check=True, capture_output=True, text=True)
print("Пайплайн подключён, аргументы проверены.")

## Доступ к весам

Добавьте `HF_TOKEN` в Secrets Colab и включите доступ для этого ноутбука. Токен не печатается. Весовые файлы FitVTON сохраняются только во временную файловую систему сессии.

In [ ]:
from google.colab import userdata
from huggingface_hub import hf_hub_download, login

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Добавьте HF_TOKEN в Secrets Colab и разрешите доступ ноутбуку.")
login(token=hf_token, add_to_git_credential=False)
for filename in ["checkpoints/default_lora_weights.safetensors", "checkpoints/texture_lora_weights.safetensors"]:
    hf_hub_download(repo_id="ZenoNing/FitVTON", filename=filename, local_dir=str(REPO), token=hf_token)
del hf_token
print("Веса FitVTON загружены. Базовая FLUX-модель загрузится при первом запуске.")

## Фотографии

Загрузите три файла: вертикальное фото одного человека в полный рост, фото пальто и фото брюк. Лучше использовать отдельные фото вещей на нейтральном фоне. Изображения отправляются в текущую Colab-сессию, а не сохраняются в этом проекте.

In [ ]:
from google.colab import files
from PIL import Image
from IPython.display import display

def upload_one(path, prompt):
    print(prompt)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Загрузите ровно один файл за раз.")
    target = Path(path)
    target.write_bytes(next(iter(uploaded.values())))
    with Image.open(target) as image:
        print("Размер:", image.size)
        display(image.copy())
    return target

person_image = upload_one("/content/person.jpg", "Загрузите фото человека в полный рост")

In [ ]:
coat_image = upload_one("/content/coat.jpg", "Загрузите фото пальто")

In [ ]:
trousers_image = upload_one("/content/trousers.jpg", "Загрузите фото брюк")

## Параметры и генерация

Выберите ближайшие категории по фигуре и росту. Это не эквивалент сантиметров или размера одежды. Для проверки чувствительности можно изменить `BODY_SHAPE` и повторить генерацию; изменение категории не означает точное моделирование S/M/L.

In [ ]:
import subprocess
import sys
from pathlib import Path
from IPython.display import display
from PIL import Image

GENDER = "male"
BODY_SHAPE = "average"
BODY_HEIGHT = "medium-tall"
OUTPUT_DIR = REPO / "outputs/free_pilot"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def generate_tryon(reference_image, garment_type, output_prefix):
    command = [
        sys.executable, str(REPO / "inference_demo.py"),
        "--person_image", str(person_image),
        "--reference_image", str(reference_image),
        "--gender", GENDER,
        "--shape", BODY_SHAPE,
        "--height", BODY_HEIGHT,
        "--length", "long-length",
        "--garment_type", garment_type,
        "--style", "untucked",
        "--width", "768",
        "--image_height", "1024",
        "--auto_resize",
        "--dtype", "fp16",
        "--num_inference_steps", "30",
        "--output_dir", str(OUTPUT_DIR),
        "--output_prefix", output_prefix,
    ]
    subprocess.run(command, cwd=REPO, check=True)
    matches = sorted(OUTPUT_DIR.glob(f"{output_prefix}_*.png"), key=lambda path: path.stat().st_mtime)
    if not matches:
        raise FileNotFoundError("FitVTON завершился без PNG-результата.")
    display(Image.open(matches[-1]))
    return matches[-1]

coat_result = generate_tryon(coat_image, "upper", "coat")
trousers_result = generate_tryon(trousers_image, "lower", "trousers")

In [ ]:
from zipfile import ZipFile

archive_path = Path("/content/fitvton-pilot-results.zip")
with ZipFile(archive_path, "w") as archive:
    archive.write(coat_result, arcname="coat.png")
    archive.write(trousers_result, arcname="trousers.png")
files.download(str(archive_path))